# Diffusion MRI data and derivatives in the Human Connectome Project

## Diffusion MRI 

Diffusion MRI uses directionally oriented gradients and a pulsed gradient scan sequence to sensitize the MRI measurement to location- and direction-specific random diffusion of water. 

This is useful because in locations where the tissue is densely packed, diffusion of water is restricted. In locations where tissue is oriented (e.g., in axonal bundles) diffusion is less restricted along the length of the bundles than across their boundaries providing a cue for computational tractography methods.

![](images/diffusion.jpg)


In [2]:
from pathlib import Path

# Make the path object:
cache_path = Path('/tmp/cache')

# Just because we have made a cache path object doesn't mean that the directory
# we made exists; here we check if it exists and make the directory if not.
if not cache_path.exists():
    cache_path.mkdir()

In [3]:
from utilities import ls, crawl
import nibabel as nib

In [4]:
from cloudpathlib import S3Path, S3Client

client = S3Client(
    local_cache_dir=cache_path,
    profile_name="hcp")

hcp_base_path = S3Path(
    "s3://hcp-openaccess/",
    client=client)

/srv/conda/envs/notebook/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
ls(hcp_base_path)

In [ ]:
# ls(hcp_base_path / "HCP_1200")

In [5]:
ls(hcp_base_path / "HCP_1200")

[S3Path('s3://hcp-openaccess/HCP_1200/100206'),
 S3Path('s3://hcp-openaccess/HCP_1200/100307'),
 S3Path('s3://hcp-openaccess/HCP_1200/100408'),
 S3Path('s3://hcp-openaccess/HCP_1200/100610'),
 S3Path('s3://hcp-openaccess/HCP_1200/101006'),
 S3Path('s3://hcp-openaccess/HCP_1200/101107'),
 S3Path('s3://hcp-openaccess/HCP_1200/101309'),
 S3Path('s3://hcp-openaccess/HCP_1200/101410'),
 S3Path('s3://hcp-openaccess/HCP_1200/101915'),
 S3Path('s3://hcp-openaccess/HCP_1200/102008'),
 S3Path('s3://hcp-openaccess/HCP_1200/102109'),
 S3Path('s3://hcp-openaccess/HCP_1200/102311'),
 S3Path('s3://hcp-openaccess/HCP_1200/102513'),
 S3Path('s3://hcp-openaccess/HCP_1200/102614'),
 S3Path('s3://hcp-openaccess/HCP_1200/102715'),
 S3Path('s3://hcp-openaccess/HCP_1200/102816'),
 S3Path('s3://hcp-openaccess/HCP_1200/103010'),
 S3Path('s3://hcp-openaccess/HCP_1200/103111'),
 S3Path('s3://hcp-openaccess/HCP_1200/103212'),
 S3Path('s3://hcp-openaccess/HCP_1200/103414'),
 S3Path('s3://hcp-openaccess/HCP_1200/10

In [ ]:
crawl(hcp_base_path / "HCP_1200" / "100206" / "T1w" / "Diffusion")

In [ ]:
diffusion_100206 = hcp_base_path / "HCP_1200" / "100206" / "T1w" / "Diffusion" / "data.nii.gz"

In [ ]:
img_100206 = nib.load(diffusion_100206.fspath)

In [ ]:
# Careful! This requires a lot of memory!
# img_100206.get_fdata()

## HCP tractometry derivatives

Tractometry assesses the properties of the white matter tissue along the major white matter bundles 

![](images/tractometry.jpg)

We have implemented a pipeline for doing this called [pyAFQ](https://yeatmanlab.github.io/pyAFQ) and we ran it on the 1,041 HCP subjects that have complete dMRI scans.

These derivatives are available through the [Open Neurodata project](https://neurodata.io/project/ocp/) (not to be confused with OpenNeuro!)

In [ ]:
from cloudpathlib import S3Path, S3Client

# Create a client that uses our cache path and that does not try to
# authenticate with S3.
client = S3Client(
    local_cache_dir=cache_path,
    no_sign_request=True)

# Now, create a cloudpath for the open neurodata's S3 bucket:
hcp_derivs_path = S3Path(
    "s3://open-neurodata/rokem/hcp1200/afq",
    client=client)

In [ ]:
crawl(hcp_derivs_path / "sub-100206")

In [ ]:
import pandas as pd

In [ ]:
tract_profiles = pd.read_csv(hcp_derivs_path / "sub-100206" / "ses-01" / "sub-100206_dwi_space-RASMM_model-CSD_desc-prob-afq_profiles.csv")

In [ ]:
tract_profiles

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(tract_profiles[tract_profiles["tractID"] == "CST_L"]["dki_fa"].values)
ax.set_xlabel("Node")
ax.set_ylabel("FA")